# 02 · KV Cache / MQA / GQA —— 开会做纪要，不用每次重讲

**家族位置**：08 生产级优化第 2 站。01 定了位置编码，本章给推理加速：自回归每步只算新 token，旧 K/V 存起来复用（KV Cache）；再把 kv 头从 H 压到 G（GQA），显存按 H/G 倍降。

**学习目标**：理解 Cache 把 O(S²) 重算降到 O(S) 增量；MHA/MQA/GQA 显存公式；缓存一致性验证；Time/内存对比。

## 1. 原理：纪要本 + 合并话筒

### 通俗理解

**一句话**：没 Cache 时每说一个新字，都要把前面所有字重算一遍（开会不记纪要，每次从头复述）；有 Cache 时旧 K/V 存着，只算新字（翻纪要接着说）。

**比喻**：MHA 是每人一个话筒（H 个 kv 头）；MQA 是全場合用一个话筒（1 个 kv 头，q 头共享）；GQA 是分 2 组各用一个（G=2）——话筒越少，纪要本越薄。

### 结构账

```
无 Cache：步 t 算 S=t 全序列 → 总量 Σt=O(S²)
有 Cache：步 t 只算 1 个新 token，拼缓存 → 总量 O(S)
KV 显存/层/token = 2·G·dk·4B；MHA(G=4)→GQA(G=2)→MQA(G=1) 逐级减半
模型：因果 toy 解码器 dim64/2层，q=4；kv=4/2/1；复制+模加混合训练 25ep
```

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, ConcatDataset
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import make_copy_data, make_modadd_data
from common.models import GQADecoder
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
Xc,yc=make_copy_data(2000,16,16,seed=0); Xm,ym=make_modadd_data(2000,16,16,seed=2)
Xv,yv=make_copy_data(300,16,16,seed=1); Xn,yn=make_modadd_data(300,16,16,seed=3)
tr=DataLoader(ConcatDataset([TensorDataset(Xc,yc),TensorDataset(Xm,ym)]),batch_size=128,shuffle=True)
va=DataLoader(ConcatDataset([TensorDataset(Xv,yv),TensorDataset(Xn,yn)]),batch_size=512)
print(f'mix train {len(tr.dataset)} / val {len(va.dataset)} | S=16 vocab=16')

## 2. 训练：三模型同混合任务 25ep（teacher forcing + 因果 mask）

In [ ]:
import torch.nn as nn
def fit_dec(m,epochs=25,lr=3e-3):
    opt=torch.optim.Adam(m.parameters(),lr=lr); crit=nn.CrossEntropyLoss(); hist=[]
    for ep in range(1,epochs+1):
        m.train(); tot=0
        for src,tgt in tr:
            logits=m(src); loss=crit(logits.reshape(-1,16),tgt.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()*len(src)
        hist.append(tot/len(tr.dataset))
        if ep in (1,5,10,15,20,25):
            m.eval();
            with torch.no_grad():
                tok=sum((m(s).argmax(-1)==t).sum().item() for s,t in va); tok_n=sum(t.numel() for s,t in va)
            print(f'ep {ep:02d} loss {hist[-1]:.3f} val-tok {tok/tok_n:.3f}',flush=True)
    return hist
models={}; hists={}
for name,kv in [('MHA',4),('GQA',2),('MQA',1)]:
    torch.manual_seed(0)
    m=GQADecoder(vocab=16,dim=64,depth=2,q_heads=4,kv_heads=kv)
    models[name]=m; hists[name]=fit_dec(m)
    print(f'{name} params={count_params(m)}',flush=True)

## 3. Cache 加速：同一步数，计时对比

In [ ]:
def bench(m,steps=32,repeats=3,use_cache=True):
    m.eval(); pre=torch.randint(0,16,(1,8))
    ts=[]
    for _ in range(repeats):
        t0=time.perf_counter(); m.generate(pre,steps,use_cache=use_cache); ts.append(time.perf_counter()-t0)
    return min(ts)
STEPS=32
rows=[]
for name,m in models.items():
    t_off=bench(m,STEPS,3,False); t_on=bench(m,STEPS,3,True)
    rows.append([name,t_off,t_on,t_off/t_on])
    print(f'{name}: no-cache {t_off:.3f}s cache {t_on:.3f}s speedup ×{t_off/t_on:.2f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.4))
x=np.arange(3); w=0.35
ax.bar(x-w/2,[r[1] for r in rows],w,label='no-cache',color='#DD8452')
ax.bar(x+w/2,[r[2] for r in rows],w,label='cache',color='#4C72B0')
for i,r in enumerate(rows): ax.text(i,r[1]+0.01,f'×{r[3]:.1f}',ha='center',fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows]); ax.set_ylabel('seconds (32 steps)')
ax.set_title('KV Cache 加速：只算新 token'); ax.legend()
plt.tight_layout(); plt.savefig(FIGS/'fig1_cache_time.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. GQA 显存：kv 头 4→2→1，纪要本逐级变薄 + 缓存一致性

In [ ]:
DK=16; LAYERS=2
mem={n: 2*(4 if n=='MHA' else 2 if n=='GQA' else 1)*DK*LAYERS*4 for n in ['MHA','GQA','MQA']}
print('KV bytes/token/layer-set:',mem)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(list(mem),list(mem.values()),color=['#4C72B0','#55A868','#DD8452'])
for i,(k,v) in enumerate(mem.items()): ax.text(i,v+8,f'{v}B',ha='center',fontsize=10)
ax.set_ylabel('bytes / token'); ax.set_title('KV 显存：MHA→GQA→MQA 逐级减半')
plt.tight_layout(); plt.savefig(FIGS/'fig2_gqa_mem.png',dpi=150,bbox_inches='tight'); plt.show()
with torch.no_grad():
    pre=torch.randint(0,16,(4,8))
    agree={}
    for n in models:
        c=models[n].prefill(pre)
        a=models[n].generate(pre,12,True,_caches=[(k.clone(),v.clone()) for k,v in c])
        b=models[n].generate(pre,12,False)
        agree[n]=(a==b).all().item()
print('cache==nocache 一致性:',agree)
fig,ax=plt.subplots(figsize=(6,2.4)); ax.axis('off')
for i,(k,v) in enumerate(agree.items()): ax.text(0.05,0.7-i*0.25,f'{k}: cache 与无 cache 生成逐 token 一致 = {bool(v)}',fontsize=11)
ax.set_title('一致性验证：Cache 只省算，不改数')
plt.tight_layout(); plt.savefig(FIGS/'fig3_consistent.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. 总结与下一步

KV Cache 把 O(S²) 重算降到 O(S) 增量（实测加速比见 fig1）；GQA 按 H/G 压显存（fig2）；一致性全 True（fig3）。下一步 `03_FlashAttention_SDPA`：注意力分块算，少搬显存。

## 附：三模型训练曲线


In [ ]:
fig,ax=plt.subplots(figsize=(6,3.2))
for n,c in [('MHA','#4C72B0'),('GQA','#55A868'),('MQA','#DD8452')]: ax.plot(hists[n],label=n,color=c)
ax.set_xlabel('epoch'); ax.set_ylabel('CE loss'); ax.set_title('混合任务训练：三模型同起点'); ax.legend()
plt.tight_layout(); plt.savefig(FIGS/'fig4_tradeoff.png',dpi=150,bbox_inches='tight'); plt.show()
print({n: round(h[-1],4) for n,h in hists.items()})